# Predicted versus actual FPL points

Select **season, gameweek and forecast time**. Compare predicted points with actual points, minutes, goals, assists and bonus. Charts and error metrics show prediction errors.

Use Run All with the project's `.venv` kernel. The notebook reads local snapshots without API calls or retraining. Defaults select the latest completed round and its latest saved pre-deadline forecast.

- **Model points** are the original saved predictions.
- **App points** reconstruct model points × availability from the same snapshot's injury flags using the current app formula. They are not a record of a user's app session.
- **Error = prediction − actual points.** Positive values indicate overprediction.
- Actual values are official player points, excluding captain multipliers, Bench Boost and transfer hits. Double-gameweek fixtures are summed.
- Evaluate only rounds marked `finished` and `data_checked`. Missing history is excluded from metrics, never replaced by zero. Available history with no fixtures in a completed round gives zero points.

One round alone provides little evidence of long-term model quality.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/capture_fpl.py').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from forecast_comparison import catalog, compare, metrics, outcome_source

pd.set_option('display.max_colwidth', 70)
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.spines.top': False, 'axes.spines.right': False})
CATALOG = catalog(ROOT)
if CATALOG.empty:
    raise ValueError('No valid local forecasts. Run src/capture_fpl.py first.')
rounds = CATALOG[['season', 'GW']].drop_duplicates().copy()
rounds['results available'] = [outcome_source(ROOT, r.season, r.GW)[1] is not None for r in rounds.itertuples()]
display(rounds.reset_index(drop=True))

## Player comparison

Select model or app points as the evaluation basis. Position and name filters apply to the table and charts; the row limit changes only the display. CSV export includes every player in the selected snapshot.

The registry includes many zero-minute reserves. Also inspect the top 25 players selected **by their predictions**, not their actual results.


In [ ]:
ready = rounds[rounds['results available']]
default = (ready if len(ready) else rounds).iloc[-1]
season_widget = widgets.Dropdown(options=sorted(CATALOG.season.unique()), value=default.season, description='Season:')
gw_widget = widgets.Dropdown(description='Gameweek:')
snapshot_widget = widgets.Dropdown(description='Forecast:', layout=widgets.Layout(width='95%'), style={'description_width': 'initial'})
score_widget = widgets.Dropdown(options=[('Model points', 'prediction'), ('App points (availability-adjusted)', 'app_prediction')], description='Points:')
position_widget = widgets.Dropdown(options=['All', 'GK', 'DEF', 'MID', 'FWD'], description='Position:')
search_widget = widgets.Text(description='Player:', placeholder='Search by name')
sort_widget = widgets.Dropdown(options=[('Highest forecast', 'score'), ('Highest actual points', 'actual_points'), ('Largest error', 'abs_error'), ('Most overestimated', 'error'), ('Most underestimated', 'under')], description='Sort:')
limit_widget = widgets.IntSlider(value=30, min=10, max=100, step=10, description='Rows:')
export_button = widgets.Button(description='Export selected GW to CSV', icon='download')
output = widgets.Output()
export_output = widgets.Output()

def set_snapshots(*_):
    subset = CATALOG[(CATALOG.season == season_widget.value) & (CATALOG.GW == gw_widget.value)]
    options = [(f"{r.forecast_at.tz_convert('Europe/Oslo'):%d %b %Y %H:%M:%S} | {r.model} | {r.snapshot}", i) for i, r in subset.iterrows()]
    snapshot_widget.options = options
    if options:
        snapshot_widget.value = options[-1][1]

def set_gameweeks(*_):
    gw_widget.options = sorted(CATALOG.loc[CATALOG.season.eq(season_widget.value), 'GW'].unique())
    gw_widget.value = int(default.GW) if default.GW in gw_widget.options else gw_widget.options[-1]
    set_snapshots()

set_gameweeks()

def selected_comparison():
    return compare(ROOT, CATALOG.loc[snapshot_widget.value])

def render(*_):
    if snapshot_widget.value is None:
        return
    with output:
        clear_output(wait=True)
        record = CATALOG.loc[snapshot_widget.value]
        full = selected_comparison()
        frame = full.copy()
        if position_widget.value != 'All':
            frame = frame[frame.position.eq(position_widget.value)]
        if search_widget.value.strip():
            frame = frame[frame.name.str.contains(search_widget.value.strip(), case=False, regex=False, na=False)]
        score = score_widget.value
        display(Markdown(f"**{record.season} · GW{record.GW}** — forecast {record.forecast_at.tz_convert('Europe/Oslo'):%d %b %Y %H:%M}, deadline {record.deadline.tz_convert('Europe/Oslo'):%d %b %Y %H:%M} (Oslo time)."))
        print('Forecast file:', Path(record.forecast_file).relative_to(ROOT))
        print('Outcome source:', full.attrs.get('outcome_source') or 'Waiting for finished, checked round')
        print('Outcomes observed:', full.attrs.get('labels_observed_at') or '—')
        if frame.empty:
            display(Markdown('No players match the filter.'))
            return
        top = frame.sort_values([score, 'player_id'], ascending=[False, True]).head(25)
        report = pd.DataFrame([metrics(frame, score), metrics(top, score)], index=['All filtered players', 'Top 25 by forecast'])
        display(report.round(3))
        frame['score'] = frame[score]
        frame['error'] = frame.score - frame.actual_points
        frame['abs_error'] = frame.error.abs()
        frame['under'] = -frame.error
        valid = frame.dropna(subset=['score', 'actual_points'])
        if valid.empty:
            display(Markdown('**Awaiting results.** Forecasts are shown, but the gameweek is not evaluated yet.'))
        else:
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            for pos, group in valid.groupby('position'):
                axes[0].scatter(group.score, group.actual_points, label=pos, alpha=.55, s=25)
            lo = min(0, valid.score.min(), valid.actual_points.min())
            hi = max(1, valid.score.max(), valid.actual_points.max()) + 1
            axes[0].plot([lo, hi], [lo, hi], '--', color='gray', label='Perfect prediction')
            axes[0].set(xlabel='Forecast', ylabel='Actual FPL points', title='Forecast vs actual')
            axes[0].legend()
            shown = frame.sort_values(['score', 'player_id'], ascending=[False, True]).head(15).iloc[::-1]
            y = np.arange(len(shown))
            axes[1].barh(y-.2, shown.score, height=.4, label='Forecast')
            axes[1].barh(y+.2, shown.actual_points, height=.4, label='Actual')
            axes[1].set(yticks=y, yticklabels=shown.name, xlabel='Points', title='15 highest rankings before the round')
            axes[1].legend()
            fig.tight_layout()
            plt.show()
            plt.close(fig)
        columns = ['player_id', 'name', 'team', 'position', 'prediction', 'app_prediction', 'actual_points', 'error', 'actual_minutes', 'goals', 'assists', 'bonus', 'clean_sheets', 'status_at_forecast', 'outcome_status']
        display(frame.sort_values([sort_widget.value, 'player_id'], ascending=[False, True], na_position='last')[columns].head(limit_widget.value).round(2).reset_index(drop=True))
        display(Markdown('`error` follows the score type selected above. `status_at_forecast`: a=available, d=doubtful, i=injured, s=suspended, u=unavailable.'))

def export_csv(_):
    record = CATALOG.loc[snapshot_widget.value]
    destination = ROOT / 'artifacts/forecast_comparisons' / f"{record.season}_GW{record.GW}_{record.snapshot}.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    data = selected_comparison()
    data['forecast_at'] = record.forecast_at.isoformat()
    data['snapshot'] = record.snapshot
    data['labels_observed_at'] = data.attrs.get('labels_observed_at')
    data.to_csv(destination, index=False)
    with export_output:
        clear_output()
        print('Saved:', destination)

season_widget.observe(set_gameweeks, names='value')
gw_widget.observe(set_snapshots, names='value')
for control in [snapshot_widget, score_widget, position_widget, search_widget, sort_widget, limit_widget]:
    control.observe(render, names='value')
export_button.on_click(export_csv)
display(widgets.VBox([widgets.HBox([season_widget, gw_widget]), snapshot_widget, widgets.HBox([score_widget, position_widget]), widgets.HBox([search_widget, sort_widget]), limit_widget, export_button, export_output, output]))
render()

## Results across completed gameweeks

Use one forecast per round: the latest saved before the deadline. Model names are shown because models may differ between rounds. This evaluates saved forecasts rather than a controlled comparison of one fixed model. The table always uses **raw model points**, independently of the selection above.


In [ ]:
summary_rows = []
for (season, gw), group in CATALOG.groupby(['season', 'GW']):
    record = group.sort_values('forecast_at').iloc[-1]
    if outcome_source(ROOT, season, gw)[1] is None:
        continue
    comparison = compare(ROOT, record)
    summary_rows.append({'season': season, 'GW': gw, 'model': record.model, 'snapshot': record.snapshot, **metrics(comparison)})
summary = pd.DataFrame(summary_rows)
if summary.empty:
    print('No finished rounds with local forecasts and verified results yet.')
else:
    display(summary.round(3))
    fig, ax = plt.subplots(figsize=(10, 4))
    x = summary.season + ' GW' + summary.GW.astype(str)
    ax.plot(x, summary.MAE, marker='o', label='MAE')
    ax.plot(x, summary.RMSE, marker='o', label='RMSE')
    ax.set(ylabel='Error in FPL points', title='Errors per finished Gameweek (lower is better)')
    ax.legend()
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Updating actual results

After a round finishes, run `.venv/bin/python src/capture_fpl.py` from the project root and rerun this notebook. Existing forecasts remain intact; the next complete capture supplies updated actual points. A round without a saved pre-deadline forecast cannot be evaluated here.
